In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F




from TtoGmodel import TextToGraphTransformer

In [2]:
from Circuits import Circuits
circuits= Circuits()

Loading dataset files...
Loaded dataset files successfully.


In [3]:
print(circuits.component_lists[0])
print(circuits.graphs[0])

['VDD', 'VSS', 'VIN1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
[[0 0 0 0 1 1 0 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 1 1 1 1]
 [1 0 0 1 0 1 0 0]
 [1 0 0 1 1 0 0 0]
 [0 0 1 1 0 0 0 0]
 [0 1 0 1 0 0 0 0]]


In [4]:
import torch
import torch.nn.functional as F

def collate_fn(batch):
    seqs, mats = zip(*batch)

    # Convert sequences to torch tensors
    seqs = [torch.tensor(seq, dtype=torch.long) for seq in seqs]

    # Convert adjacency matrices (NumPy -> PyTorch)
    mats = [torch.tensor(mat, dtype=torch.float32) for mat in mats]

    # Get max sizes
    max_seq_len = max(len(seq) for seq in seqs)
    max_nodes = max(mat.size(0) for mat in mats)

    # Pad sequences
    padded_seqs = torch.stack([
        F.pad(seq, (0, max_seq_len - len(seq)), value=0)
        for seq in seqs
    ])

    # Pad adjacency matrices
    padded_mats = torch.stack([
        F.pad(mat, (0, max_nodes - mat.size(1), 0, max_nodes - mat.size(0)), value=0)
        for mat in mats
    ])

    seq_lengths = torch.tensor([len(seq) for seq in seqs])

    return padded_seqs, padded_mats, seq_lengths


In [5]:
dataset = list(zip(circuits.component_indices, circuits.graphs))
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)


In [7]:
print(circuits.component_indices[0])
print(circuits.graphs[0])


[150, 149, 27, 215, 71, 716, 867, 496]
[[0 0 0 0 1 1 0 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 1 1 1 1]
 [1 0 0 1 0 1 0 0]
 [1 0 0 1 1 0 0 0]
 [0 0 1 1 0 0 0 0]
 [0 1 0 1 0 0 0 0]]


In [8]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm  # for progress bar

# Initialize model parameters
vocab_size = len(circuits.vocab)  # Number of unique components
embedding_dim = 128
hidden_dim = 256
num_heads = 8
num_layers = 4
dropout = 0.1

# Initialize the model
model = TextToGraphTransformer(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout
)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

# Use prepared DataLoader
train_loader = loader

# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for input_seqs, adj_mats, seq_lengths in progress_bar:
        # Move data to device
        input_seqs = input_seqs.to(device)
        adj_mats = adj_mats.to(device)
        seq_lengths = seq_lengths.to(device)

        optimizer.zero_grad()

        # Forward pass
        predicted_adj = model(input_seqs, adj_mats, seq_lengths)

        # Compute loss for adjacency matrix (e.g., using Mean Squared Error)
        loss = F.mse_loss(predicted_adj, adj_mats)

        # Backward and optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")

# Save model
torch.save(model.state_dict(), 'TextToGraphTransformer.pth')


c:\Python313\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Epoch 1/10:   0%|          | 0/105 [00:00<?, ?it/s]C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (torch.Size([32, 233, 233])) that is different to the input size (torch.Size([233, 233])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(predicted_adj, adj_mats)
Epoch 1/10:   1%|          | 1/105 [00:00<01:06,  1.57it/s, loss=5.97]C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (torch.Size([32, 230, 230])) that is different to the input size (torch.Size([230, 230])). This will likely lead to incorrect results due to broadcasting. Please ensur

Epoch 1/10 - Avg Loss: 362.2155


Epoch 2/10:  18%|█▊        | 19/105 [00:01<00:05, 15.51it/s, loss=0.0213]C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (torch.Size([32, 190, 190])) that is different to the input size (torch.Size([190, 190])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(predicted_adj, adj_mats)
Epoch 2/10:  31%|███▏      | 33/105 [00:02<00:04, 15.60it/s, loss=0.0236]C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (torch.Size([32, 177, 177])) that is different to the input size (torch.Size([177, 177])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(predicted_adj, adj_mats)
Epoch 2/10:  49%|████▊     | 51/105 [00:03<00:03, 14.14it/s, loss=0.0188]C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size 

Epoch 2/10 - Avg Loss: 0.0236


Epoch 3/10:   6%|▌         | 6/105 [00:00<00:06, 14.50it/s, loss=0.021] C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (torch.Size([32, 192, 192])) that is different to the input size (torch.Size([192, 192])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(predicted_adj, adj_mats)
Epoch 3/10:  10%|▉         | 10/105 [00:00<00:06, 15.56it/s, loss=0.02]  C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (torch.Size([32, 195, 195])) that is different to the input size (torch.Size([195, 195])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(predicted_adj, adj_mats)
Epoch 3/10:  27%|██▋       | 28/105 [00:01<00:04, 15.47it/s, loss=0.0266]C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (

...

Epoch 8/10 - Avg Loss: 0.0231


Epoch 9/10:  76%|███████▌  | 80/105 [00:05<00:01, 14.74it/s, loss=0.0215]C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (torch.Size([32, 173, 173])) that is different to the input size (torch.Size([173, 173])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(predicted_adj, adj_mats)
Epoch 9/10:  99%|█████████▉| 104/105 [00:06<00:00, 14.88it/s, loss=0.022] C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (torch.Size([21, 206, 206])) that is different to the input size (torch.Size([206, 206])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(predicted_adj, adj_mats)
Epoch 9/10: 100%|██████████| 105/105 [00:06<00:00, 15.06it/s, loss=0.0224]


Epoch 9/10 - Avg Loss: 0.0229


Epoch 10/10:  99%|█████████▉| 104/105 [00:06<00:00, 15.36it/s, loss=0.023] C:\Users\pasin\AppData\Local\Temp\ipykernel_29632\3487950567.py:54: UserWarning: Using a target size (torch.Size([21, 204, 204])) that is different to the input size (torch.Size([204, 204])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(predicted_adj, adj_mats)
Epoch 10/10: 100%|██████████| 105/105 [00:06<00:00, 15.07it/s, loss=0.0251]

Epoch 10/10 - Avg Loss: 0.0229


In [21]:
def evaluate_adjacency_matrix(model, input_seq, vocab, device, threshold=0.5):
    model.eval()
    with torch.no_grad():
        # Convert to tensor and move to device
        input_tensor = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)  # Shape: (1, seq_len)
        
        # Dummy adjacency matrix (not used in current context, just for input)
        seq_len = input_tensor.size(1)
        dummy_adj = torch.zeros((1, seq_len, seq_len), dtype=torch.float).to(device)
        dummy_lengths = torch.tensor([seq_len]).to(device)

        # Get model output
        predicted_adj = model(input_tensor, dummy_adj, dummy_lengths)  # (1, seq_len, seq_len)

        # Print raw output (predicted values before thresholding)
        #print("\nRaw Predicted Adjacency Matrix (N x N):")
        #print(predicted_adj.squeeze(0).cpu().numpy())  # Show the raw predicted values

        # Apply sigmoid to normalize the values to the [0, 1] range
        adjacency_matrix = torch.sigmoid(predicted_adj.squeeze(0))  # Apply sigmoid for probabilities

        # Threshold to get binary values (0 or 1)
        adjacency_matrix_binary = (adjacency_matrix > threshold).float()  # Apply threshold (default 0.5)

        # Print the thresholded adjacency matrix
        print("\nPredicted Adjacency Matrix (Binary, N x N):")
        print(adjacency_matrix_binary.cpu().numpy())  # Removing the batch dimension and moving to CPU


In [28]:
ex_index = 2  # Example index for the input sequence
sample_input = circuits.component_indices[ex_index]  # Use any valid component sequence
print("Sample Input Sequence:", circuits.component_lists[ex_index])
print("Sample adjacency matrix:")
print(circuits.graphs[ex_index])
evaluate_adjacency_matrix(model, sample_input, circuits.vocab, device)


Sample Input Sequence: ['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Sample adjacency matrix:
[[0 0 0 0 0 0 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 1 1 0 0 0 0 0]
 [0 0 0 1 0 1 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 1 1 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 1 0 0 0 1 0 0 0 0 0]
 [1 0 0 0 0 1 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 1 1 1]
 [0 0 0 1 0 0 1 0 0 0 1 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 1 0 0 0 1]
 [0 1 0 0 0 0 0 0 0 0 1 0 0 1 0]]

Predicted Adjacency Matrix (Binary, N x N):
[[1. 0. 1. 0. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 0.]
 [0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 1.]
 [1. 0. 1. 0. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 0.]
 [0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 1.]
 [1. 0. 1. 0. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 0.]
 [1. 0. 1. 0. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 0.

In [ ]:
# Define the file path to save the model and hyperparameters
save_path = "TtoGmodel_checkpoint.pth"

# Create a dictionary to store the model state and hyperparameters
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'embed_dim': embed_dim,
    'num_heads': num_heads,
    'num_layers': num_layers,
    'dropout': dropout,
    'learning_rate': learning_rate,
    'text_vocab_size': text_vocab_size,
    'graph_input_dim': graph_input_dim,
}

# Save the checkpoint
torch.save(checkpoint, save_path)
print(f"Model and hyperparameters saved to {save_path}")

Model and hyperparameters saved to model_checkpoint.pth


In [ ]:
# Load the saved model checkpoint
checkpoint_path = "TtoGmodel_checkpoint.pth"
checkpoint = torch.load(checkpoint_path)

# Reinitialize the model with the saved hyperparameters
model = TexttoGraphTransformer(
    graph_input_dim=checkpoint['graph_input_dim'],
    text_vocab_size=checkpoint['text_vocab_size'],
    embed_dim=checkpoint['embed_dim'],
    num_heads=checkpoint['num_heads'],
    num_layers=checkpoint['num_layers'],
    dropout=checkpoint['dropout']
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Select a sample graph input
sample_graph = graph_data[0].unsqueeze(0).to(device)  # Add batch dimension

# Generate a sequence
start_token = torch.tensor([82], dtype=torch.long).to(device)  # Assuming 0 is the start token
generated_sequence = [start_token.item()]

for _ in range(5000):  # Generate up to 50 tokens
    input_sequence = torch.tensor(generated_sequence, dtype=torch.long).unsqueeze(0).to(device)
    output = model(sample_graph, input_sequence)
    next_token = torch.argmax(output[:, -1, :], dim=-1).item()  # Get the most probable next token
    generated_sequence.append(next_token)
    if next_token == 892:  # Assuming 892 is the end token
        break

# Convert indices back to components
generated_text = circuits.get_component_fromlist(generated_sequence)

print("Generated Sequence:", generated_text)

NameError: name 'GtoTmodel' is not defined